# CPTS 440 — Chess Agent Demo

This notebook demonstrates our depth-limited minimax search engine with material evaluation.

**What you'll see:**
1. Engine overview, having it choose moves and reporting search statistics
2. Tactical puzzle solving (mate-in-1, winning material)
3. Depth comparison, how deeper search finds better moves at higher cost
4. A full AI-vs-AI game with move-by-move visualization

In [ ]:
import chess
import chess.svg
from IPython.display import SVG, display, HTML

from src import engine
from src.engine import play_game

---
## 1. Engine Overview

Our agent uses **depth-limited minimax** search with a **material-based evaluation function**.

- **Search**: Explores the game tree to a fixed depth, alternating between maximizing (White) and minimizing (Black) players
- **Evaluation**: Scores positions in centipawns based on material balance (P=100, N=320, B=330, R=500, Q=900)
- **Move ordering**: Captures and checks are searched first to improve pruning in future iterations

Let's give the engine a position and see what it finds.

In [ ]:
board = chess.Board()

result = engine.choose_move(board, depth=2)

print(f"Best move : {result.move}")
print(f"Eval score: {result.score:+.0f} centipawns")
print(f"Nodes     : {result.nodes:,}")
print(f"Depth     : {result.depth} plies")

In [ ]:
# Show the board BEFORE and AFTER the engine's chosen move, side by side
before_svg = chess.svg.board(
    board, size=350,
    arrows=[chess.svg.Arrow(result.move.from_square, result.move.to_square, color="#cc220080")],
)

board_after = board.copy()
board_after.push(result.move)
after_svg = chess.svg.board(
    board_after, size=350,
    fill={result.move.to_square: "#88cc44"},
)

display(HTML(f"""
<div style='display:flex; gap:2em; align-items:center'>
  <div><h4>Before</h4>{before_svg}</div>
  <div style='font-size:2em'>→</div>
  <div><h4>After {result.move}</h4>{after_svg}</div>
</div>
"""))

---
## 2. Tactical Puzzles

Can the engine solve simple tactics? Let's test it on known positions.

In [ ]:
def show_puzzle(title: str, fen: str, expected_uci: str, depth: int = 3) -> None:
    """Display a puzzle position, let the engine solve it, and show the result."""
    board = chess.Board(fen)
    result = engine.choose_move(board, depth=depth)

    found = result.move.uci() if result.move else "(none)"
    match = "✅" if found == expected_uci else "❌"

    board_after = board.copy()
    if result.move:
        board_after.push(result.move)

    before_svg = chess.svg.board(
        board, size=300,
        arrows=(
            [chess.svg.Arrow(result.move.from_square, result.move.to_square, color="#cc220080")]
            if result.move else []
        ),
    )
    after_svg = chess.svg.board(board_after, size=300)

    display(HTML(f"""
    <h3>{title} {match}</h3>
    <p>Engine chose <b>{found}</b> (expected <b>{expected_uci}</b>)
    — eval: {result.score:+.0f} cp, nodes: {result.nodes:,}, depth: {result.depth}</p>
    <div style='display:flex; gap:1.5em; align-items:center'>
      <div>{before_svg}</div>
      <div style='font-size:1.5em'>→</div>
      <div>{after_svg}</div>
    </div>
    """))

In [ ]:
# Puzzle 1: White to move with mate in 1 by Qh5#
show_puzzle(
    title="Mate in 1",
    fen="rnbqkbnr/pppp1ppp/8/4p3/6P1/5P2/PPPPP2P/RNBQKBNR b KQkq - 0 2",
    expected_uci="d8h4",
    depth=3,
)

In [ ]:
# Puzzle 2: White to move with win by capturing Black's hanging queen
show_puzzle(
    title="Win the queen",
    fen="r1bqkbnr/pppppppp/2n5/8/3NP3/8/PPP2PPP/RNBQKB1R w KQkq - 1 3",
    expected_uci="d4c6",
    depth=2,
)

In [ ]:
# Puzzle 3: White to move with back-rank mate by Qe8# (rook guards, king trapped)
show_puzzle(
    title="Back-rank mate",
    fen="6k1/5ppp/8/8/8/8/1Q3PPP/6K1 w - - 0 1",
    expected_uci="b2b8",
    depth=3,
)

---
## 3. Depth Comparison

How does increasing search depth affect the engine's choice and the cost (nodes searched)?

We run the same position at depth 1, 2, and 3 and compare.

In [ ]:
fen = "r1bqkb1r/pppppppp/2n2n2/8/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 2 3"
board = chess.Board(fen)

print(f"Position: {fen}\n")
print(f"{'Depth':<8} {'Best Move':<12} {'Score (cp)':<12} {'Nodes':>10}")
print("-" * 46)

for d in range(1, 4):
    result = engine.choose_move(board, depth=d)
    move_str = result.move.uci() if result.move else "(none)"
    print(f"{d:<8} {move_str:<12} {result.score:+<12.0f} {result.nodes:>10,}")

SVG(chess.svg.board(board, size=350))

Notice how:
- **Node count grows exponentially** with depth (branching factor ~30–35 in the middlegame)
- **Deeper search can change the best move** as the engine sees further consequences
- This motivates alpha-beta pruning (Week 3) to search deeper within the same node budget

---
## 4. AI vs. AI Game

Let's run a full game where the engine plays both sides, then visualize the result.

In [ ]:
game = play_game(depth=2, max_moves=80)

print(f"Game result : {game.result}")
print(f"Total plies : {len(game.plies)}")
print(f"Total nodes : {sum(p.nodes for p in game.plies):,}")

In [ ]:
# Show the evaluation score over the course of the game
print(f"{'Ply':<6} {'Move':<10} {'Score (cp)':>10} {'Nodes':>8}  {'Bar'}")
print("-" * 60)

for i, ply in enumerate(game.plies):
    # Simple text bar chart: each █ = 50 centipawns
    bar_len = int(ply.score / 50)
    if bar_len >= 0:
        bar = " " * 20 + "█" * min(bar_len, 20)
    else:
        filled = min(abs(bar_len), 20)
        bar = " " * (20 - filled) + "█" * filled
    print(f"{i+1:<6} {ply.move_uci:<10} {ply.score:>+10.0f} {ply.nodes:>8,}  |{bar}|")

In [ ]:
# Visualize key moments: first move, a midgame position, and the final position
snapshots = []
indices = [0, len(game.plies) // 2, len(game.plies) - 1]

for idx in indices:
    ply = game.plies[idx]
    b = chess.Board(ply.fen_before)
    move = chess.Move.from_uci(ply.move_uci)
    svg = chess.svg.board(
        b, size=280,
        arrows=[chess.svg.Arrow(move.from_square, move.to_square, color="#cc220080")],
    )
    snapshots.append(f"""
    <div style='text-align:center'>
      <h4>Ply {idx + 1}: {ply.move_uci}</h4>
      <p>Eval: {ply.score:+.0f} cp</p>
      {svg}
    </div>
    """)

display(HTML(f"<div style='display:flex; gap:1em; flex-wrap:wrap'>{''.join(snapshots)}</div>"))

In [ ]:
# Show the final board position
final_board = chess.Board(game.final_fen)
display(HTML(f"<h3>Final Position — Result: {game.result}</h3>"))
SVG(chess.svg.board(final_board, size=400))